# Phase 2 — Dataset Preparation
This phase prepares a leakage-aware dataset manifest. Model training remains out of scope here.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
root = Path.cwd()
paths = sorted(root.rglob('*.jpeg'))
print(f'Images available for preparation: {len(paths):,}')

Images available for preparation: 8,648


## Build a manifest
Labels and split names are derived from the real folder paths.

In [2]:
label_map = {'ok_front': 'OK', 'def_front': 'Defective'}
manifest = pd.DataFrame({'path': [str(p) for p in paths]})
manifest['label'] = manifest['path'].map(lambda p: label_map[Path(p).parent.name])
manifest['split'] = manifest['path'].map(lambda p: 'train' if 'train' in Path(p).parts else ('test' if 'test' in Path(p).parts else 'unsplit'))
manifest['dataset'] = manifest['path'].map(lambda p: 'casting_data' if 'casting_data' in Path(p).parts else 'casting_512x512')
display(manifest.groupby(['dataset','split','label']).size().rename('images').to_frame())

images
dataset         split   label            
casting_512x512 unsplit Defective     781
                        OK            519
casting_data    test    Defective     453
                        OK            262
                train   Defective    3758
                        OK           2875

## Audit duplicate leakage
Exact hashes are used to flag duplicates crossing train and test boundaries.

In [3]:
def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()
manifest['sha256'] = manifest['path'].map(digest)
display(manifest[manifest.duplicated('sha256', False)].groupby(['dataset','split']).size().rename('duplicate_files').to_frame())

duplicate_files
dataset      split                 
casting_data test                64
             train               64

## Phase 2 handoff
Resolve duplicate train/test groups before training and retain the provided test set for final evaluation.

In [4]:
leakage = manifest[manifest.duplicated('sha256', False)]
cross_split = leakage.groupby('sha256')['split'].nunique()
print(f'Hashes duplicated across multiple splits: {(cross_split > 1).sum():,}')
manifest.to_csv('dataset_manifest_phase2.csv', index=False)
print('Phase 2 preparation scaffold created; no model training was run.')

Hashes duplicated across multiple splits: 64
Phase 2 preparation scaffold created; no model training was run.
